# Task 4 Gen-2 (V4): Deterministic De-Rotation + Nightmare-V4 + SigLIP Contrastive ViT (Colab GPU)

Standalone Colab GPU notebook for **Phase 3 Gen-2 Vision Training** (`Zero Interference` with running `t4_hard_outputs/` / `eva02l`):
- **Outputs directory:** `/content/drive/MyDrive/t4_gen2_v4_outputs`
- **New Visual Robustness Upgrades:**
  1. **Deterministic Single-Image De-Rotation (`detect_and_derotate_pil`)**: Straightens tilted frames with black corner triangles (`[-45°, +45°]`) with zero test parameter fitting.
  2. **Nightmare-V4 Corruptions**: `Rot±45°`, `RandomErasing/Cutout`, `Pixelation (0.35x–0.70x)`, `Gaussian Blur`, `Grayscale/Saturation + Neon RGB`, `Exposure/Fog`, `Noise + JPEG`, `MixUp/CutMix`, and `Model EMA (0.997)`.
  3. **Vision-Language Contrastive Diversity**: `siglip_l16` (`vit_large_patch16_siglip_384` @ `192×384`) or `siglip_so400m` (`vit_so400m_patch14_siglip_384` @ `196×392`).

In [ ]:
import os, subprocess, torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
!pip install -q --upgrade 'timm>=1.0.22' 'scikit-learn>=1.5'

In [ ]:
# Clone/pull cpe342 repo (or use uploaded train_t4_gen2_v4.py) & unpack task4_colab_pack.tar
REPO_DIR = '/content/cpe342-machine-learning-2026'
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/Palapluem/cpe342-machine-learning-2026.git {REPO_DIR} || true
else:
    !git -C {REPO_DIR} pull --ff-only || true

DATA_DIR = '/content/t4_data'
PACK_CANDIDATES = [
    '/content/task4_colab_pack.tar',
    '/content/drive/MyDrive/task4_colab_pack.tar',
    '/content/drive/MyDrive/cpe342/task4_colab_pack.tar',
]
if not os.path.isfile(os.path.join(DATA_DIR, 'folds_sgkf.npy')):
    pack = next((p for p in PACK_CANDIDATES if os.path.isfile(p)), None)
    assert pack is not None, f'Upload task4_colab_pack.tar to one of: {PACK_CANDIDATES}'
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Unpacking:', pack)
    !tar -xf '{pack}' -C '{DATA_DIR}'
print('Data ready:', os.listdir(DATA_DIR))

In [ ]:
# Choose model: 'siglip_l16' (fast on T4/L4) or 'siglip_so400m' (recommended on L4/A100)
MODEL_KEY = 'siglip_l16'
OUT_DIR = '/content/drive/MyDrive/t4_gen2_v4_outputs'
SCRIPT_PATH = '/content/cpe342-machine-learning-2026/project/task4/scripts/train_t4_gen2_v4.py'
assert os.path.isfile(SCRIPT_PATH), f'Missing script: {SCRIPT_PATH}'
os.makedirs(OUT_DIR, exist_ok=True)
print('Target model:', MODEL_KEY, '| Out dir:', OUT_DIR)

In [ ]:
# Run 5-fold SGKF training with atomic epoch checkpointing & automatic resume
!python3 -u '{SCRIPT_PATH}' \
    --data-dir '{DATA_DIR}' \
    --out-dir '{OUT_DIR}' \
    --models '{MODEL_KEY}' \
    --num-workers 2 \
    --resume

In [ ]:
# Verify completed Gen-2 zip archive ready to download
import zipfile
zp = os.path.join(OUT_DIR, f't4_gen2_{MODEL_KEY}.zip')
if os.path.isfile(zp):
    with zipfile.ZipFile(zp) as zf:
        print('Ready zip:', zp, f'({os.path.getsize(zp)/1024**2:.2f} MB)', zf.namelist())
else:
    print('Partial folds saved so far:', sorted(os.listdir(os.path.join(OUT_DIR, f'folds_{MODEL_KEY}'))))